<!-- Imports and Setup -->
<!-- to load tools -->

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, confusion_matrix, classification_report
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore') # Keeps your notebook clean from warnings

<!-- Data Loading & Stratified Split -->
<!-- the data preparation -->

In [ ]:
# Cell 2: Data Prep
# NOTE: To run the other dataset, change to 'Fraud_Data.csv' and 'class'
DATA_FILE = 'creditcard.csv' 
TARGET_COL = 'Class'

print(f"Loading {DATA_FILE}...")
df = pd.read_csv(DATA_FILE)

# Separate features and target
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Fraud cases in test set: {sum(y_test == 1)}")

Baseline Model (Logistic Regression)
This trains your interpretable baseline model and evaluates it using Cross-Validation.

In [ ]:
# Cell 3: Baseline Model
print("Training Baseline: Logistic Regression...")

# Initialize with balanced class weights
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Set up Stratified K-Fold for reliable cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Run Cross-Validation
scoring_metrics = ['f1', 'average_precision']
cv_results_lr = cross_validate(log_reg, X_train, y_train, cv=cv, scoring=scoring_metrics)

print(f"CV F1-Score: {np.mean(cv_results_lr['test_f1']):.4f} (+/- {np.std(cv_results_lr['test_f1']):.4f})")
print(f"CV AUC-PR: {np.mean(cv_results_lr['test_average_precision']):.4f} (+/- {np.std(cv_results_lr['test_average_precision']):.4f})")

# Fit on the full training set and predict on the holdout test set
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

<!-- Ensemble Model (XGBoost) with Basic Tuning
This builds your advanced model and handles the heavy imbalance mathematically. -->

In [ ]:
# Cell 4: XGBoost Ensemble
print("Training Ensemble: XGBoost...")

# Calculate scale_pos_weight to handle extreme imbalance
neg_class = y_train.value_counts()[0]
pos_class = y_train.value_counts()[1]
scale_weight = neg_class / pos_class

xgb_model = xgb.XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=42)

# Basic Hyperparameter Grid (Kept small so it runs relatively quickly in a notebook)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5]
}

# Grid Search with CV
grid_search = GridSearchCV(
    estimator=xgb_model, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='average_precision', # Optimize for AUC-PR
    n_jobs=-1 # Uses all available CPU cores
)

grid_search.fit(X_train, y_train)
best_xgb = grid_search.best_estimator_

print(f"Best XGBoost Params: {grid_search.best_params_}")

# Predict on holdout test set
y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

Final Evaluation & Model Comparison
This generates the side-by-side comparison table you need for your deliverables.

In [ ]:
# Cell 5: Model Evaluation Table
print("--- Final Test Set Evaluation ---")
results = []

for model_name, y_pred, y_proba in [("Logistic Regression", y_pred_lr, y_proba_lr), 
                                    ("XGBoost", y_pred_xgb, y_proba_xgb)]:
    
    auc_pr = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    results.append({
        "Model": model_name,
        "AUC-PR": round(auc_pr, 4),
        "F1-Score": round(f1, 4),
        "True Negatives": cm[0,0],
        "False Positives": cm[0,1],
        "False Negatives": cm[1,0],
        "True Positives": cm[1,1]
    })

# Display as a clean Pandas DataFrame for your report
results_df = pd.DataFrame(results)
display(results_df)